# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fkashaf19-afk/ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
import os

if os.path.basename(os.getcwd()) != "ml-internship":
    if not os.path.exists("ml-internship"):
        !git clone https://github.com/fkashaf19-afk/ml-internship.git
    %cd ml-internship

Cloning into 'ml-internship'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 130 (delta 39), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 1.87 MiB | 7.71 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/ml-internship


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring** (with a classification-style proxy label underneath).

The ML-02 decision was: given a limited weekly review budget (~50 pages), which pages
should a content editor look at first? That's not a yes/no call — it's an *ordering*
problem. A pure classifier only tells you in/out; it doesn't tell the editor what to
open first when the queue is longer than the budget. So the output has to be a
priority score per `content_id`, sorted descending, and handed to the editor as a
ranked queue.

Why not the other three:
- **Clustering** would group content types by behavior, but it doesn't produce an
  action-ordered list — no "do this one first."
- **Plain classification** (declining / not declining) collapses everything past the
  threshold into one bucket, losing the within-bucket ordering the editor actually needs.
- A **scoring model** trained toward the proxy label (below) is the mechanism; ranking
  is how its output gets consumed.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label` — a defined proxy, not an observed future outcome.**

`is_declining_label = (trend_direction == "down")`, and `trend_direction` is itself
computed from `trend_pct`. That means the label comes from a rule applied to the
*current* 90-day snapshot, not from watching what actually happened to a page in a
later time window — the starter CSV is a single trailing-90-day cross-section, so
there's no forward panel here to observe a true future outcome against.

That's a real limitation, and I'm naming it rather than hiding it: a genuinely
observed target (e.g. "did traffic actually recover 60 days after a refresh") would
need the warehouse's `fact_content_daily_performance` time series, which is out of
scope for this notebook. For now, `is_declining_label` is the best available proxy —
used honestly, as a proxy, with a leakage rule attached:

**Leakage rule:** `trend_direction` and `trend_pct` define the label. They can never
be used as *features* — doing so would let the model "predict" its own definition.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# is_declining_label is not a raw column -- it's derived from trend_direction,
# exactly as the starter pipeline (01_prepare_features.py) computes it.
df['is_declining_label'] = df['trend_direction'] == 'down'

print("Target column sketch: is_declining_label\n")
print(df['is_declining_label'].value_counts())
print("\nShare declining (base rate):", round(df['is_declining_label'].mean(), 3))
df[['content_id', 'trend_direction', 'is_declining_label']].head(10)

Target column sketch: is_declining_label

is_declining_label
True     16262
False    13738
Name: count, dtype: int64

Share declining (base rate): 0.542


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,True
1,content_a1fb4e703a9e,down,True
2,content_9aa793d4d895,down,True
3,content_331d6c4de07b,stable,False
4,content_d99b7a2d90ca,down,True
5,content_d4084a4bc775,down,True
6,content_9a34b442b552,down,True
7,content_a63219c6e95a,stable,False
8,content_5e6c160719bc,down,True
9,content_c27558df2b0c,down,True


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: precision@50.**

The editor's weekly budget is ~50 pages (from ML-02). Precision@50 answers the exact
question that matters: of the top 50 pages the ranking hands to the editor, how many
are genuinely worth reviewing? That's a direct match to how a reviewer actually works
down a ranked list — not plain accuracy over all 30,000 rows, most of which the editor
will never look at this week.

It's also defensible against a baseline: the base rate is 54.2% declining, so any
ranking worth using needs to clear that bar comfortably in the top 50 — and the
starter pipeline already shows this is achievable (see Section 5).

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
# One row = one content page (content_id), scored and ranked individually --
# not a client-level or time-period aggregate.

# trend_direction / trend_pct are excluded on purpose: they define the label,
# so they are never features (see Section 2's leakage rule).
leakage_cols = ['trend_direction', 'trend_pct']

candidate_cols = [
    'content_id', 'client_id', 'content_type',
    'ctr', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'avg_position', 'word_count', 'is_declining_label'
]
lane_cols = [c for c in candidate_cols if c in df.columns]

unit_df = df[lane_cols].copy()

print("Rows (= content pages):", len(unit_df))
print("Unique content_id:", unit_df['content_id'].nunique())
print("Unique client_id:", unit_df['client_id'].nunique())
unit_df.head(10)

Rows (= content pages): 30000
Unique content_id: 30000
Unique client_id: 32


,content_id,client_id,content_type,ctr,engagement_rate,scroll_rate,ai_traffic_pct,avg_position,word_count,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,0.76,5.88,4.55,0.0,10.6,3221.0,True
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0.05,0.00,10.00,0.0,20.3,2481.0,True
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.09,0.00,28.57,0.0,36.5,3515.0,True
3,content_331d6c4de07b,client_19581e27de,keyword article,0.49,1.28,3.45,0.0,6.2,NaN,False
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.13,0.00,24.29,0.0,44.0,2803.0,True
5,content_d4084a4bc775,client_f369cb89fc,keyword article,0.03,0.00,25.00,0.0,8.5,3080.0,True
6,content_9a34b442b552,client_8722616204,keyword article,0.00,0.00,0.00,0.0,7.0,3059.0,True
7,content_a63219c6e95a,client_19581e27de,keyword article,0.06,3.57,7.14,0.0,21.2,NaN,False
8,content_5e6c160719bc,client_6208ef0f77,keyword article,0.09,5.88,6.25,0.0,46.0,3807.0,True
9,content_c27558df2b0c,client_19581e27de,keyword article,0.16,0.00,0.00,0.0,4.9,NaN,True


In [4]:
# Sanity checks against the known data gotchas, so the frame above isn't just
# asserted -- it's checked against the actual file.

if 'avg_position' in unit_df.columns:
    zero_position = (unit_df['avg_position'] == 0).sum()
    print("Rows with avg_position == 0 (means 'no data', not rank zero):", zero_position)

if 'ai_traffic_pct' in unit_df.columns:
    over_100 = (unit_df['ai_traffic_pct'] > 100).sum()
    print("Rows with ai_traffic_pct > 100 (expected -- different measurement systems):", over_100)

if 'content_type' in unit_df.columns:
    print("\nMissingness by content_type (word_count):")
    print(unit_df.assign(has_word_count=unit_df.get('word_count').notna() if 'word_count' in unit_df else None)
          .groupby('content_type')['is_declining_label'].count())

Rows with avg_position == 0 (means 'no data', not rank zero): 1205
Rows with ai_traffic_pct > 100 (expected -- different measurement systems): 23

Missingness by content_type (word_count):
content_type
comparison article      697
feedly article         2096
keyword article       27207
Name: is_declining_label, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

No single signal cleanly separates "review this first" from "leave it": `ctr`,
`engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `avg_position`, and `word_count`
each carry partial, weakly-correlated evidence, and they interact — a low CTR page
with high engagement is a different story than a low CTR, low engagement page, and a
fixed if/else can't hold that many conditional branches without turning into an
unmaintainable rule pile that still misses cases.

This isn't hypothetical for this lane — the starter pipeline (already run, see
`outputs/model_report.md`) shows the actual gap:

- Baseline fixed rule: precision@50 = 0.240 (~12 of the top 50 are real)
- Random forest: precision@50 = 0.740 (~37 of the top 50 are real)

That's roughly a 3x jump from letting a model combine the signals instead of a
hand-written rule choosing one or two thresholds. That gap is the whole justification
for spending the next several weeks on this rather than shipping a simple filter.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.